In [1]:
!pip install google-genai google-adk feedparser requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 1.2 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=e24cbc96472b460dc96df6346c3245988f902388cf09f7fec10afac35c09b5b2
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your_gemini_api_key_here from AI Studio"
os.environ["GOOGLE_CHAT_WEBHOOK"] = "your_webhook_url_here"

In [4]:
from google import genai

client = genai.Client()
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="สวัสดี บอกฉันสั้นๆ ว่าคุณคือใคร"
)
print(response.text)

สวัสดีครับ! ผมคือ AI (ปัญญาประดิษฐ์) ที่พัฒนาโดย Google พร้อมช่วยเหลือและตอบคำถามคุณในทุกๆ เรื่องครับ


In [8]:
import requests
import xml.etree.ElementTree as ET

def fetch_threat_news(source: str = "cisa") -> str:
    """ดึงข่าวความปลอดภัยไซเบอร์ด้วย requests โดยตรง"""
    feeds = {
        "cisa": "https://www.cisa.gov/cybersecurity-advisories/all.xml",
        "hackernews": "https://feeds.feedburner.com/TheHackersNews"
    }
    url = feeds.get(source, feeds["cisa"])

    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=10)

    if response.status_code != 200:
        return f"ดึงข้อมูลไม่ได้ (status: {response.status_code})"

    # Parse XML
    root = ET.fromstring(response.content)
    namespace = {"atom": "http://www.w3.org/2005/Atom"}

    news_list = []

    # ลอง RSS format ก่อน
    items = root.findall(".//item")
    if items:
        for item in items[:3]:
            title = item.findtext("title", "ไม่มีชื่อ")
            desc = item.findtext("description", "")[:200]
            news_list.append(f"- {title}: {desc}")
    else:
        # ลอง Atom format
        entries = root.findall(".//atom:entry", namespace)
        for entry in entries[:3]:
            title = entry.findtext("atom:title", "ไม่มีชื่อ", namespace)
            summary = entry.findtext("atom:summary", "")[:200]
            news_list.append(f"- {title}: {summary}")

    if not news_list:
        return "ไม่พบข่าวในขณะนี้"

    return "\n".join(news_list)

# ทดสอบ
result = fetch_threat_news("cisa")
print(result)

- CubeSpace CW0057 Reaction Wheel: <p><a href="https://github.com/cisagov/CSAF/blob/develop/csaf_files/OT/white/2026/icsa-26-183-02.json"><strong>View CSAF</strong></a></p>
<h2>Summary</h2>
<p><strong>Successful exploitation of this vu
- Gardyn IoT Hub: <p><a href="https://github.com/cisagov/CSAF/blob/develop/csaf_files/OT/white/2026/icsa-26-183-03.json"><strong>View CSAF</strong></a></p>
<h2>Summary</h2>
<p><strong>Successful exploitation of these v
- ST Engineering iDirect iQ-Series Terminals: <p><a href="https://github.com/cisagov/CSAF/blob/develop/csaf_files/OT/white/2026/icsa-26-183-01.json"><strong>View CSAF</strong></a></p>
<h2>Summary</h2>
<p><strong>Successful exploitation of these v


In [9]:
import re

def clean_html(text: str) -> str:
    """ลบ HTML tags ออก"""
    return re.sub(r'<[^>]+>', '', text).strip()

def analyze_and_summarize(raw_news: str) -> str:
    """ให้ Gemini วิเคราะห์และสรุปข่าวเป็นภาษาไทย"""
    clean_news = clean_html(raw_news)  # ทำความสะอาดก่อน

    prompt = f"""
คุณคือผู้เชี่ยวชาญด้านความปลอดภัยไซเบอร์
จากข่าวด้านล่าง กรุณา:
1. แปลชื่อช่องโหว่และอธิบายเป็นภาษาไทย
2. สรุปสั้นๆ 3 บรรทัด สำหรับผู้บริหาร
3. ระบุระดับความเสี่ยง: สูง / กลาง / ต่ำ

ข่าว:
{clean_news}

ตอบเป็นภาษาไทยเท่านั้น
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

# ทดสอบ
news = fetch_threat_news("cisa")
summary = analyze_and_summarize(news)
print(summary)

ในฐานะผู้เชี่ยวชาญด้านความปลอดภัยไซเบอร์ ขอรายงานวิเคราะห์ช่องโหว่จากข่าวดังกล่าว ดังนี้ครับ:

---

### 1. แปลชื่อช่องโหว่และอธิบายเป็นภาษาไทย

*   **CubeSpace CW0057 Reaction Wheel (ช่องโหว่ในระบบล้อควบคุมการทรงตัวของดาวเทียม CubeSpace รุ่น CW0057)**
    *   **อธิบาย:** ช่องโหว่นี้เกิดขึ้นกับอุปกรณ์ฮาร์ดแวร์ที่ใช้ควบคุมทิศทางและการทรงตัวของดาวเทียมขนาดเล็ก (CubeSat) หากผู้โจมตีเจาะระบบสำเร็จ จะสามารถเข้าควบคุมทิศทางของดาวเทียม ทำให้ดาวเทียมหลุดวงโคจร หรือไม่สามารถทำงานตามภารกิจได้
*   **Gardyn IoT Hub (ช่องโหว่ในกล่องควบคุมอุปกรณ์ IoT ของ Gardyn)**
    *   **อธิบาย:** ช่องโหว่ในอุปกรณ์ควบคุมระบบปลูกพืชอัจฉริยะภายในบ้าน หากถูกโจมตีสำเร็จ ผู้ประสงค์ร้ายอาจใช้เป็นช่องทางในการเจาะเข้าสู่เครือข่าย Wi-Fi ภายในบ้านเพื่อขโมยข้อมูล หรือเข้าควบคุมการทำงานของอุปกรณ์อัจฉริยะอื่นๆ
*   **ST Engineering iDirect iQ-Series Terminals (ช่องโหว่ในสถานีปลายทางรับส่งสัญญาณดาวเทียม iDirect ตระกูล iQ-Series)**
    *   **อธิบาย:** ช่องโหว่ในอุปกรณ์รับส่งสัญญาณดาวเทียมที่ใช้ในภาคธุรกิจและการทหาร หากถูกโจมตีสำเ

In [10]:
import requests

def send_to_google_chat(message: str) -> dict:
    """ส่งข่าวสรุปเข้า Google Chat Webhook"""
    webhook_url = os.environ["GOOGLE_CHAT_WEBHOOK"]
    payload = {
        "cards": [{
            "header": {
                "title": "🛡️ CTIA Daily Alert",
                "subtitle": "Cyber Threat Intelligence Agent"
            },
            "sections": [{
                "widgets": [{
                    "textParagraph": {"text": message}
                }]
            }]
        }]
    }
    response = requests.post(webhook_url, json=payload)
    return {"status": response.status_code, "ok": response.ok}

# ทดสอบส่งข้อความเข้า Google Chat
result = send_to_google_chat(summary)
print(result)

{'status': 200, 'ok': True}


In [11]:
def run_ctia_agent(source: str = "cisa"):
    print(f"🤖 CTIA Agent กำลังทำงาน... แหล่งข่าว: {source}")
    print("=" * 50)

    # Step 1: ดึงข่าว
    print("📡 Step 1: กำลังดึงข่าวจาก CISA...")
    raw_news = fetch_threat_news(source)
    print("✅ ดึงข่าวสำเร็จ\n")

    # Step 2: วิเคราะห์
    print("🧠 Step 2: กำลังวิเคราะห์ด้วย Gemini...")
    summary = analyze_and_summarize(raw_news)
    print("✅ วิเคราะห์สำเร็จ\n")

    # Step 3: ส่ง
    print("📨 Step 3: กำลังส่งเข้า Google Chat...")
    result = send_to_google_chat(summary)
    print(f"✅ ส่งสำเร็จ! Status: {result}\n")

    print("=" * 50)
    print("🎯 CTIA Agent ทำงานเสร็จสมบูรณ์!")
    return summary

# รันเลย!
final_output = run_ctia_agent("cisa")

🤖 CTIA Agent กำลังทำงาน... แหล่งข่าว: cisa
📡 Step 1: กำลังดึงข่าวจาก CISA...
✅ ดึงข่าวสำเร็จ

🧠 Step 2: กำลังวิเคราะห์ด้วย Gemini...
✅ วิเคราะห์สำเร็จ

📨 Step 3: กำลังส่งเข้า Google Chat...
✅ ส่งสำเร็จ! Status: {'status': 200, 'ok': True}

🎯 CTIA Agent ทำงานเสร็จสมบูรณ์!
